<a href="https://colab.research.google.com/github/jaihong1027/Python_Hangman/blob/main/Hangman%E5%90%8A%E6%AD%BB%E9%AC%BC%E5%92%8C%E6%96%B7%E9%A0%AD%E5%8F%B0_%E7%B6%93%E5%85%B8%E7%8C%9C%E5%AD%97%E9%81%8A%E6%88%B2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random, sys, time

# 擴充的吊死鬼圖案（10階段）
HANGMAN_PICS = [r'''
  +---+
      |
      |
      |
     ===
''', r'''
  +---+
  O   |
      |
      |
     ===
''', r'''
  +---+
  O   |
  |   |
      |
     ===
''', r'''
  +---+
  O   |
 /|   |
      |
     ===
''', r'''
  +---+
  O   |
 /|\  |
      |
     ===
''', r'''
  +---+
  O   |
 /|\  |
 /    |
     ===
''', r'''
  +---+
  O   |
 /|\  |
 / \  |
     ===

''', r'''
  +---+
 [O   |
 /|\  |
 / \  |
     ===
''', r'''
  +---+
 [O]  |
 /|\  |
 / \  |
     ===
''', r'''
  +---+
 [O]  |
 /|\  |
_/ \  |
     ===
''', r'''
  +---+
 [O]  |
 /|\  |
_/ \_  |
     ===
''']

# 類別與單字資料 (統一為大寫以確保比較一致性)
WORD_CATEGORIES = {
    'Animals': 'ALLIGATOR ANT BABOON BADGER BAT BEAR BEAVER CAMEL CAT CLAM COBRA COUGAR COYOTE CROW DEER DOG DONKEY DUCK EAGLE ELEPHANT FERRET FOX FROG GIRAFFE GOAT GOOSE GORILLA HAWK HIPPOPOTAMUS JELLYFISH KANGAROO LEOPARD LION LIZARD LLAMA MOLE MONKEY MOOSE MOUSE MULE NARWHAL NEWT OCTOPUS OTTER OWL PANDA PARROT PENGUIN PIGEON PYTHON RABBIT RAM RAT RAVEN RHINO RHINOCEROS SALMON SEAL SHARK SHEEP SKUNK SLOTH SNAKE SPIDER SQUIRREL STORK SWAN TIGER TOAD TROUT TURKEY TURTLE VAMPIRE BAT WEASEL WHALE WOLF WOMBAT ZEBRA'.split(),
    'Countries': 'TAIWAN JAPAN CANADA BRAZIL FRANCE GERMANY EGYPT CHILE KENYA NORWAY SWEDEN FINLAND POLAND TURKEY INDIA CHINA KOREA MEXICO SPAIN ITALY KYRGYZSTAN NETHERLANDS UZBEKISTAN TURKMENISTAN PHILIPPINES SEYCHELLES LIECHTENSTEIN LUXEMBOURG'.split(),
}
VOWELS='AEIOU'

# 主遊戲函式
def main():
    print('Hangman, by Al Sweigart al@inventwithpython.com')
    print('--- Enhanced Version ---')

    category = None
    difficulty = None
    words = []

    while not words:
        # 類別選擇
        category = chooseCategory()
        current_category_words = WORD_CATEGORIES[category]

        # 難度選擇
        difficulty = chooseDifficulty()
        words = filterWordsByDifficulty(current_category_words, difficulty)

        if not words:
            print("No words found for the selected category and difficulty combination. Please try again.")

    # 設置新遊戲變數
    missedletters = []
    correctletters = []
    secretWord = random.choice(words)
    hintUsed = False
    startTime = time.time()

    while True:
        drawHangman(missedletters, correctletters, secretWord, category)

        # 顯示提示選項
        if not hintUsed and len(missedletters) < len(HANGMAN_PICS) - 2:
            print('Type "?" to use a hint (costs one miss).')

        guess = getPlayerGuess(missedletters + correctletters)

        if guess == '?':
            if not hintUsed and len(missedletters) < len(HANGMAN_PICS) - 2:
                giveHint(secretWord, correctletters)
                missedletters.append('?')
                hintUsed = True
                continue
            else:
                print('Hint unavailable.')
                continue

        if guess in secretWord:
                correctletters.append(guess)
        else:
            missedletters.append(guess)

        # 檢查勝利
        if all(letter in correctletters  for letter in secretWord):
            # 勝利時不需再調用 drawHangman，因為 printVictory 內部會處理
            printVictory(secretWord, startTime)
            break

        # 檢查失敗
        if len(missedletters) >= len(HANGMAN_PICS) - 1:
            # 失敗時不需再調用 drawHangman，因為 printDefeat 內部會處理
            printDefeat(secretWord, startTime)
            break

# 類別選擇
def chooseCategory():
    print('Choose a category(Enter the number):')
    for i, cat in enumerate(WORD_CATEGORIES.keys(), 1):
        print(f'{i}. {cat}')
    while True:
        choice = input('> ')
        if choice.isdigit() and 1 <= int(choice) <= len(WORD_CATEGORIES):
            return list(WORD_CATEGORIES.keys())[int(choice) - 1]
        else:
            print('Invalid choice.')

# 難度選擇
def chooseDifficulty():
    print('Choose difficulty: 1-Easy 2-Medium 3-Hard')
    while True:
        choice = input('> ')
        if choice in ['1', '2', '3']:
            return int(choice)
        else:
            print('Invalid choice.')

# 根據難度過濾單字
def filterWordsByDifficulty(words, level):
    if level == 1:
        return [w for w in words if len(w) <= 5]
    elif level == 2:
        return [w for w in words if 5 < len(w) <= 7]
    else:
        return [w for w in words if len(w) > 7]

# 提示功能 (隨機選擇未猜出的唯一字母)
def giveHint(secretWord, correctletters):
    ungussed_letters = [letter for letter in set(secretWord) if letter not in correctletters]
    if ungussed_letters:
        hint_letter = random.choice(ungussed_letters)
        print(f'Hint: The word contains "{hint_letter}"')
    else:
        print('Hint: All unique letters have been revealed.')

# 繪製遊戲狀態 (錯誤字母顯示紅色，正確字母顯示綠色)
def drawHangman(missedletters, correctletters, secretWord, category):

    # ANSI 顏色代碼
    GREEN_START = '\033[92m'  # 綠色
    RED_START = '\033[91m'    # 紅色
    COLOR_END = '\033[0m'     # 重置

    print(HANGMAN_PICS[len(missedletters)])
    print('Category:', category)

    # 顯示錯誤字母 (Missed letters) - 帶顏色
    colored_missed_letters = []
    for letter in missedletters:
        # 將每個錯誤字母顯示為紅色
        colored_missed_letters.append(f'{RED_START}{letter}{COLOR_END}')

    print('Missed letters:', ' '.join(colored_missed_letters))

    # 顯示秘密單字 (Secret Word) - 猜中顯示綠色
    print('Secret Word:', end=' ')
    for letter in secretWord:
        if letter in correctletters:
            # 正確猜中的字母顯示為綠色
            print(f'{GREEN_START}{letter}{COLOR_END}', end=' ')
        else:
            # 未猜中的字母顯示為底線
            print('_', end=' ')

    print('\n')

# 玩家輸入
def getPlayerGuess(alreadyGuessed):
    while True:
        guess = input('Guess a letter:\n> ').upper()
        if guess == '?':
            return guess
        elif len(guess) != 1 or not guess.isalpha():
            print('Please enter a single LETTER.')
        elif guess in alreadyGuessed:
            print('Already guessed.')
        else:
            return guess

# 勝利訊息 (所有文字逐字列印，秘密單字綠色)
def printVictory(secretWord, startTime):

    GREEN_START = '\033[92m'
    COLOR_END = '\033[0m'

    # 1. 逐字列印 "You WON!"
    print('\n', end='', flush=True)
    victory_msg = "You WON!"
    for c in victory_msg:
        print(c, end='', flush=True)
        time.sleep(0.1)
    print('!')

    # 2. 逐字列印秘密單字 (綠色特效)
    colored_message = f'The word was "{GREEN_START}{secretWord}{COLOR_END}"'

    for c in colored_message:
        print(c, end='', flush=True)
        time.sleep(0.05)

    print(f'\nTime: {int(time.time() - startTime)} seconds')

# 失敗訊息 (圖案閃爍，文字逐字列印，秘密單字綠色)
def printDefeat(secretWord, startTime):

    GREEN_START = '\033[92m'
    COLOR_END = '\033[0m'

    # 1. 圖案閃爍特效
    final_pic_index = len(HANGMAN_PICS) - 1

    for _ in range(3): # 閃爍 3 次
        # 顯示最後圖案
        print(HANGMAN_PICS[final_pic_index])
        time.sleep(0.2)

        # 清空圖案位置 (使用足夠多的換行符號來「推開」舊圖案)
        print('\n' * (HANGMAN_PICS[0].count('\n') + 2))
        time.sleep(0.1)

    # 重新繪製最後的圖案
    print(HANGMAN_PICS[final_pic_index])

    # 2. 逐字列印 "You LOST!"
    print('\n', end='', flush=True)
    defeat_msg = "You LOST!"
    for c in defeat_msg:
        print(c, end='', flush=True)
        time.sleep(0.1)
    print('!')

    # 3. 逐字列印秘密單字 (綠色特效)
    colored_message = f'The word was "{GREEN_START}{secretWord}{COLOR_END}"'

    for c in colored_message:
        print(c, end='', flush=True)
        time.sleep(0.05)

    print(f'\nTime: {int(time.time() - startTime)} seconds')

# 程式進入點
if __name__ == '__main__':
    try:
        main()
    except KeyboardInterrupt:
        sys.exit()

Hangman, by Al Sweigart al@inventwithpython.com
--- Enhanced Version ---
Choose a category(Enter the number):
1. Animals
2. Countries
> 1
Choose difficulty: 1-Easy 2-Medium 3-Hard
> .3
Invalid choice.
> 3

  +---+
      |
      |
      |
     ===

Category: Animals
Missed letters: 
Secret Word: _ _ _ _ _ _ _ _ _ _ _ _ 

Type "?" to use a hint (costs one miss).
Guess a letter:
> a

  +---+
      |
      |
      |
     ===

Category: Animals
Missed letters: 
Secret Word: _ _ _ _ _ _ _ _ A _ _ _ 

Type "?" to use a hint (costs one miss).
Guess a letter:
> e

  +---+
  O   |
      |
      |
     ===

Category: Animals
Missed letters: E
Secret Word: _ _ _ _ _ _ _ _ A _ _ _ 

Type "?" to use a hint (costs one miss).
Guess a letter:
> i

  +---+
  O   |
      |
      |
     ===

Category: Animals
Missed letters: E
Secret Word: _ I _ _ _ _ _ _ A _ _ _ 

Type "?" to use a hint (costs one miss).
Guess a letter:
> o

  +---+
  O   |
      |
      |
     ===

Category: Animals
Missed letters: E
Se